In [0]:
from pyspark.sql.functions import rand,floor,col,when,monotonically_increasing_id,lit

In [0]:
data1 =[
    (1,'a','Kol'),(2,'b','Kol'),(3,'c','Kol'),(4,'d','Kol'),(5,'e','Kol'),(6,'f','Kol'),(7,'g','Kol'),(8,'h','Kol'),(9,'i','Mum'),(10,'j','Mum'),]
schema1 = ["id","name","city"]
df1=spark.createDataFrame(data1,schema1)

data2 =[
    (1,'aa','Kol'),(2,'bb','Kol'),(3,'cc','Del'),(4,'dd','Del'),(5,'ee','Amd'),(6,'ff','Amd'),(7,'gg','Blr'),(8,'hh','Blr'),(9,'ii','Mum'),(10,'jj','Mum'),]
schema2=["id","scope","city"]
df2=spark.createDataFrame(data2,schema2)

display(df1)
display(df2)

In [0]:
# selecting the number of keys causing skewness ... Here for demo I have selected 1
salt_offset=1
df_sel=df1.groupBy("city").count().orderBy(col("count").desc()).head(salt_offset)
print(df_sel)

In [0]:
# Here splitting the skewed data into n buckets ... Here for demo I have selected 3
salt_bucket=3
df1_salted=df1.withColumn("salt",when(col("city")==df_sel[0]["city"],monotonically_increasing_id()%salt_bucket+1).otherwise(-1))

display(df1_salted)

In [0]:
df2_selected_rows=df2.where(col("city")==df_sel[0]["city"])
df2_salted=df2_selected_rows.withColumn("salt",lit(1))

for salt in range(2,salt_bucket+1):
    df2_salted=df2_salted.union(df2_selected_rows.withColumn("salt",lit(salt)))

#extract & Combine the non skewed data 
df2_salted=df2_salted.union(df2.where(col("city")!=df_sel[0]["city"]).withColumn("salt",lit(-1)))
display(df2_salted)